# Imports

In [ ]:
!pip install rasterio

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc

from pathlib import Path
import rasterio
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision import transforms
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import StepLR


# Constants

In [ ]:
DATASET_DIR = Path("/kaggle/input/cloud-masking-dataset/content/train")
IMG_DIR =   DATASET_DIR / "data"
MASK_DIR =   DATASET_DIR / "masks"
MODEL_PATH = "/kaggle/working/best_cloud_net.pth"

In [ ]:
img_names = [img_name.name for img_name in IMG_DIR.rglob("*")]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
len(img_names)

## Modeling

In [ ]:
class CloudNet(nn.Module):
    def __init__(self, input_rows=512, input_cols=512, num_of_channels=4, num_of_classes=1):
        super(CloudNet, self).__init__()
        # Encoder
        self.conv1 = nn.Sequential(
            nn.Conv2d(num_of_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )
        # Decoder
        self.upconv4 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2, padding=0)
        self.dec4 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),  # 128 (skip) + 128 (upconv)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.upconv3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2, padding=0)
        self.dec3 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.upconv2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2, padding=0)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.upconv1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2, padding=0)
        self.dec1 = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.final = nn.Conv2d(16, num_of_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        conv1 = self.conv1(x)  # [B, 16, 512, 512]
        pool1 = self.pool(conv1)  # [B, 16, 256, 256]
        conv2 = self.conv2(pool1)  # [B, 32, 256, 256]
        pool2 = self.pool(conv2)  # [B, 32, 128, 128]
        conv3 = self.conv3(pool2)  # [B, 64, 128, 128]
        pool3 = self.pool(conv3)  # [B, 64, 64, 64]
        conv4 = self.conv4(pool3)  # [B, 128, 64, 64]
        pool4 = self.pool(conv4)  # [B, 128, 32, 32]
        # Bottleneck
        bottleneck = self.bottleneck(pool4)  # [B, 256, 32, 32]
        # Decoder
        up4 = self.upconv4(bottleneck)  # [B, 128, 64, 64]
        up4 = torch.cat([up4, conv4], dim=1)  # [B, 256, 64, 64]
        dec4 = self.dec4(up4)  # [B, 128, 64, 64]
        up3 = self.upconv3(dec4)  # [B, 64, 128, 128]
        up3 = torch.cat([up3, conv3], dim=1)  # [B, 128, 128, 128]
        dec3 = self.dec3(up3)  # [B, 64, 128, 128]
        up2 = self.upconv2(dec3)  # [B, 32, 256, 256]
        up2 = torch.cat([up2, conv2], dim=1)  # [B, 64, 256, 256]
        dec2 = self.dec2(up2)  # [B, 32, 256, 256]
        up1 = self.upconv1(dec2)  # [B, 16, 512, 512]
        up1 = torch.cat([up1, conv1], dim=1)  # [B, 32, 512, 512]
        dec1 = self.dec1(up1)  # [B, 16, 512, 512]
        final = self.final(dec1)  # [B, 1, 512, 512]
        return torch.sigmoid(final)

def model_arch(input_rows=512, input_cols=512, num_of_channels=4, num_of_classes=1):
    return CloudNet(input_rows, input_cols, num_of_channels, num_of_classes)

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = pred.contiguous().view(-1)
        target = target.contiguous().view(-1)
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        return 1 - dice

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_names, transform=None):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        self.img_names = img_names
        self.transform = transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = self.img_dir / img_name
        mask_path = self.mask_dir / img_name

        try:
            with rasterio.open(img_path) as src:
                image = src.read()
                if image.shape[0] != 4:
                    raise ValueError(f"Image {img_name} has {image.shape[0]} channels, expected 4")
                image = image.astype(np.float16) / 65535.0
                image = image.transpose(1, 2, 0)

            try:
                with rasterio.open(mask_path) as src:
                    mask = src.read(1)
            except:
                mask = np.array(Image.open(mask_path))
                if len(mask.shape) == 3:
                    mask = mask[:, :, 0]
            mask = (mask > 0).astype(np.float16)

            if self.transform:
                image = self.transform(image)
                mask = torch.from_numpy(mask).float()
            else:
                image = torch.from_numpy(image.transpose(2, 0, 1)).float()
                mask = torch.from_numpy(mask).float()

            if image.shape[0] != 4:
                raise ValueError(f"Image {img_name} tensor has shape {image.shape}, expected (4, H, W)")

            return image, mask.unsqueeze(0)

        except Exception as e:
            print(f"Error loading {img_name}: {str(e)}")
            dummy_image = torch.zeros((4, 512, 512), dtype=torch.float16)
            dummy_mask = torch.zeros((1, 512, 512), dtype=torch.float16)
            return dummy_image, dummy_mask

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, device, accum_steps=2):
    scaler = GradScaler()
    best_val_dice = 0.0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_dice = 0.0
        optimizer.zero_grad()
        
        for i, (images, masks) in enumerate(train_loader):
            images, masks = images.to(device), masks.to(device)
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks) / accum_steps
            scaler.scale(loss).backward()
            if (i + 1) % accum_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            train_loss += loss.item() * accum_steps * images.size(0)
            pred = (outputs > 0.5).float()
            intersection = (pred * masks).sum()
            dice = (2. * intersection + 1e-6) / (pred.sum() + masks.sum() + 1e-6)
            train_dice += dice.item() * images.size(0)

        train_loss /= len(train_loader.dataset)
        train_dice /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        val_dice = 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, masks)
                val_loss += loss.item() * images.size(0)
                pred = (outputs > 0.5).float()
                intersection = (pred * masks).sum()
                dice = (2. * intersection + 1e-6) / (pred.sum() + masks.sum() + 1e-6)
                val_dice += dice.item() * images.size(0)

        val_loss /= len(val_loader.dataset)
        val_dice /= len(val_loader.dataset)

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), MODEL_PATH)

        # Step the scheduler to update the learning rate
        scheduler.step()

        torch.cuda.empty_cache()

In [ ]:
def compute_dice(pred, target, smooth=1e-6):
    """Compute Dice coefficient between predicted and target masks."""
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    return dice

def infer_and_evaluate(model, data_loader, device, output_dir):
    model.eval()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Access the dataset's img_names from the data_loader
    dataset = data_loader.dataset
    img_names = dataset.img_names

    total_dice = 0.0
    num_samples = 0
    batch_start_idx = 0

    with torch.no_grad():
        for images, masks in data_loader:
            images, masks = images.to(device), masks.to(device)
            with autocast():
                pred_masks = model(images)  # [B, 1, H, W], sigmoid output in [0, 1]
            
            # Convert to binary mask
            pred_masks = (pred_masks > 0.5).float()  # Threshold at 0.5

            # Save each predicted mask as uint8 TIFF with the same name as the input image
            for i in range(images.size(0)):
                # Get the filename for the current image using the batch index
                img_idx = batch_start_idx + i
                mask_filename = output_dir / Path(img_names[img_idx]).name  # Keep original extension (e.g., .tif)

                # Convert mask to uint8 NumPy array (0 or 255)
                mask_uint8 = pred_masks[i].squeeze(0)  # Remove channel dim: [1, H, W] -> [H, W]
                mask_uint8 = (mask_uint8 * 255).clamp(0, 255).to(torch.uint8).cpu().numpy()

                # Save as TIFF using rasterio
                with rasterio.open(
                    mask_filename,
                    'w',
                    driver='GTiff',
                    height=mask_uint8.shape[0],
                    width=mask_uint8.shape[1],
                    count=1,
                    dtype='uint8',
                    crs=None,
                    transform=None
                ) as dst:
                    dst.write(mask_uint8, 1)

                # Compute Dice coefficient
                dice = compute_dice(pred_masks[i], masks[i])
                total_dice += dice.item()
                num_samples += 1

            # Update the batch start index
            batch_start_idx += images.size(0)

            # Clear memory after each batch
            torch.cuda.empty_cache()
            gc.collect()

    # Compute average Dice coefficient
    avg_dice = total_dice / num_samples if num_samples > 0 else 0.0
    print(f"Average Dice Coefficient on test set: {avg_dice:.4f}")
    return avg_dice

In [ ]:
# Clear GPU memory at start
gc.collect()
torch.cuda.empty_cache()
print(f"Initial GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB allocated, "
      f"{torch.cuda.memory_reserved() / 1024**3:.2f} GiB reserved")

In [ ]:
test_split = 0.15 
val_split = 0.2

# Split into training and test sets
train_img_names, test_img_names = train_test_split(
    img_names, test_size=test_split, random_state=42
)
# Split into training+validation and test sets
train_val_img_names, test_img_names = train_test_split(
        img_names, test_size=test_split, random_state=42
    )
print(f"Train+Val images: {len(train_val_img_names)}, Test images: {len(test_img_names)}")

    # Further split train+validation into train and validation
train_img_names, val_img_names = train_test_split(
        train_val_img_names, test_size=val_split , random_state=42
    )
print(f"Training images: {len(train_img_names)}, Validation images: {len(val_img_names)}")

In [ ]:
batch_size = 4
learning_rate = 1e-3
num_epochs = 10
train_split = 0.8
accum_steps = 2 # Simulate batch size of 4 (2 * 2)

train_transform = transforms.Compose([
        transforms.ToTensor(),
        # transforms.RandomHorizontalFlip(p=0.5),
        # transforms.Resize((256, 256)),
    ])
val_transform = transforms.Compose([
        transforms.ToTensor(),
        # transforms.Resize((256, 256)),
    ])

# Datasets and DataLoaders
train_dataset = CustomDataset(IMG_DIR, MASK_DIR, train_img_names, train_transform)
val_dataset = CustomDataset(IMG_DIR, MASK_DIR, val_img_names, val_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# Model, loss, optimizer
model = model_arch(num_of_channels=4, num_of_classes=1).to(device)
criterion = DiceLoss()
optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
exp_lr_scheduler = StepLR(optimizer_ft, step_size=30, gamma=0.1)

# Train the model
train_model(model, train_loader, val_loader, criterion, optimizer_ft, exp_lr_scheduler, num_epochs, device, accum_steps)

In [ ]:
#To continue training ..
train_model(model, train_loader, val_loader, criterion, optimizer_ft, exp_lr_scheduler, num_epochs, device, accum_steps)

In [ ]:
model_path = "/kaggle/input/best_model/pytorch/default/1/best_cloud_net_0.8816.pth"
output_dir = "/kaggle/working/first"

# Create dataset for test images only
dataset = CustomDataset(IMG_DIR,MASK_DIR, test_img_names, val_transform)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# Check model memory usage
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")
print(f"After model load: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB allocated, "
      f"{torch.cuda.memory_reserved() / 1024**3:.2f} GiB reserved")

# Run inference on test set
infer_and_evaluate(model, data_loader, device, output_dir)

In [ ]:
model_loaded = model_arch(num_of_channels=4, num_of_classes=1).to(device)
state_dict = torch.load(model_path, map_location=device,weights_only = True)
model_loaded.load_state_dict(state_dict)

# Run inference on test set
infer_and_evaluate(model_loaded, data_loader, device, output_dir)

In [ ]:
# torch.cuda.empty_cache()torch.save(model.state_dict(), "/kaggle/working/cloud_net_8815.pth")

In [ ]:
model_path = "/kaggle/input/best_model/pytorch/default/1/best_cloud_net_0.8816.pth"
output_dir = "/kaggle/working/ouptput/"
batch_size = 4
test_transform = transforms.Compose([
        transforms.ToTensor(),
        # transforms.Resize((256, 256)),
    ])

test_dataset = CustomDataset(IMG_DIR, MASK_DIR, test_img_names, test_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)


model_loaded = model_arch(num_of_channels=4, num_of_classes=1).to(device)
state_dict = torch.load(model_path, map_location=device,weights_only = True)
model_loaded.load_state_dict(state_dict)

# infer_and_evaluate(model_loaded, test_loader, device, output_dir)

In [ ]:
import shutil
from pathlib import Path

output_dir = Path('/kaggle/working/output')  # Adjust to your actual output_dir
zip_name = '/kaggle/working/masks'  # Output ZIP file (without .zip extension)

# Create a ZIP archive of the output_dir
shutil.make_archive(zip_name, 'zip', output_dir)

print(f"ZIP file created: {zip_name}.zip")

In [ ]:
import shutil
from pathlib import Path
import os

# Adjust these to match your test dataset
mask_dir = Path('/kaggle/input/cloud-masking-dataset/content/train/masks')  # Update to your actual mask_dir
test_img_names = test_dataset.img_names  # From your test dataset
output_zip = '/kaggle/working/original_masks'  # Output ZIP file (without .zip extension)
temp_dir = Path('/kaggle/working/temp_original_masks')

# Create temporary directory to collect masks
temp_dir.mkdir(parents=True, exist_ok=True)

# Copy original masks to temporary directory
for img_name in test_img_names:
    mask_path = mask_dir / img_name
    if mask_path.exists():
        shutil.copy(mask_path, temp_dir / img_name)
    else:
        print(f"Warning: Mask not found for {img_name}")

# Create ZIP archive of the temporary directory
shutil.make_archive(output_zip, 'zip', temp_dir)

# Clean up temporary directory
shutil.rmtree(temp_dir)

print(f"ZIP file created: {output_zip}.zip")

In [ ]:
def rle_encode(mask):
    """
    Encodes a binary mask using Run-Length Encoding (RLE).
    
    Args:
        mask (np.ndarray): 2D binary mask (0s and 1s).
    
    Returns:
        str: RLE-encoded string.
    """
    pixels = mask.flatten(order='F')  # Flatten in column-major order
    pixels = np.concatenate([[0], pixels, [0]])  # Add padding to detect transitions
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1  # Get transition indices
    runs[1::2] -= runs[::2]  # Compute run lengths
    runs[::2] -= 1  # Make it 0-indexed instead of 1-indexed
    return " ".join(map(str, runs))  # Convert to string format
    

def generate_rle_csv(mask_dir, test_img_names, output_csv):
    """
    Reads masks from mask_dir, applies RLE encoding, and saves to a CSV file.
    
    Args:
        mask_dir (Path): Directory containing the original masks.
        test_img_names (list): List of filenames in the test dataset.
        output_csv (str): Path to the output CSV file.
    """
    mask_dir = Path(mask_dir)
    output_csv = Path(output_csv)
    
    # Ensure output directory exists
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    
    # List to store results
    results = []
    
    for img_name in test_img_names:
        mask_path = mask_dir / img_name
        try:
            # Try loading with rasterio
            try:
                with rasterio.open(mask_path) as src:
                    mask = src.read(1)  # Read single channel
            except:
                # Fallback to PIL
                mask = np.array(Image.open(mask_path))
                if len(mask.shape) == 3:
                    mask = mask[:, :, 0]  # Take first channel if RGB
            
            # Convert to binary mask (0s and 1s)
            mask = (mask > 0).astype(np.uint8)
            
            # Apply RLE encoding
            rle = rle_encode(mask)
            
            # Use filename without extension as ID
            img_id = Path(img_name).stem
            
            # Append to results
            results.append({'id': img_id, 'segmentation': rle})
        
        except Exception as e:
            print(f"Error processing {img_name}: {str(e)}")
            # Append empty RLE for missing/invalid masks
            results.append({'id': Path(img_name).stem, 'segmentation': ''})
    
    # Save to CSV
    df = pd.DataFrame(results, columns=['id', 'segmentation'])
    df.to_csv(output_csv, index=False)
    print(f"CSV file saved: {output_csv}")

In [ ]:
print("HI")

In [ ]:
mask_dir = '/kaggle/working/output' # Update to your actual mask_dir
test_img_names = test_dataset.img_names  # From your test dataset
output_csv = '/kaggle/working/output.csv'

generate_rle_csv(mask_dir, test_img_names, output_csv)

In [ ]:
mask_dir = '/kaggle/input/cloud-masking-dataset/content/train/masks' # Update to your actual mask_dir
test_img_names = test_dataset.img_names  # From your test dataset
output_csv = '/kaggle/working/real.csv'

generate_rle_csv(mask_dir, test_img_names, output_csv)

In [ ]:
def compute_dice(pred, target, smooth=1e-6):
    """Compute Dice coefficient between predicted and target masks."""
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    return dice

def rle_encode(mask):
    """
    Encodes a binary mask using Run-Length Encoding (RLE).
    
    Args:
        mask (np.ndarray): 2D binary mask (0s and 1s).
    
    Returns:
        str: RLE-encoded string.
    """
    pixels = mask.flatten(order='F')  # Flatten in column-major order
    pixels = np.concatenate([[0], pixels, [0]])  # Add padding to detect transitions
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1  # Get transition indices
    runs[1::2] -= runs[::2]  # Compute run lengths
    runs[::2] -= 1  # Make it 0-indexed instead of 1-indexed
    return " ".join(map(str, runs))  # Convert to string format

def infer_and_evaluate(model, data_loader, device, output_dir):
    model.eval()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Access the dataset's img_names and mask_dir from the data_loader
    dataset = data_loader.dataset
    img_names = dataset.img_names
    mask_dir = Path(dataset.mask_dir)

    total_dice = 0.0
    num_samples = 0
    batch_start_idx = 0
    pred_results = []  # For predicted masks
    true_results = []  # For original masks

    with torch.no_grad():
        for images, masks in data_loader:
            images, masks = images.to(device), masks.to(device)
            with autocast():
                pred_masks = model(images)  # [B, 1, H, W], sigmoid output in [0, 1]
            
            # Convert to binary mask
            pred_masks = (pred_masks > 0.5).float()  # Threshold at 0.5

            # Process each mask pair
            for i in range(images.size(0)):
                # Get the filename for the current image using the batch index
                img_idx = batch_start_idx + i
                img_name = img_names[img_idx]
                img_id = Path(img_name).stem  # Filename without extension

                # Process predicted mask
                pred_mask_binary = pred_masks[i].squeeze(0).cpu().numpy()  # [1, H, W] -> [H, W]
                pred_mask_binary = (pred_mask_binary > 0).astype(np.uint8)  # Ensure binary (0, 1)
                pred_rle = rle_encode(pred_mask_binary)
                pred_results.append({'id': img_id, 'segmentation': pred_rle})

                # Process original mask (from dataset)
                true_mask_binary = masks[i].squeeze(0).cpu().numpy()  # [1, H, W] -> [H, W]
                true_mask_binary = (true_mask_binary > 0).astype(np.uint8)  # Ensure binary (0, 1)
                true_rle = rle_encode(true_mask_binary)
                true_results.append({'id': img_id, 'segmentation': true_rle})

                # Compute Dice coefficient
                dice = compute_dice(pred_masks[i], masks[i])
                total_dice += dice.item()
                num_samples += 1

            # Update the batch start index
            batch_start_idx += images.size(0)

            # Clear memory after each batch
            torch.cuda.empty_cache()
            gc.collect()

    # Compute average Dice coefficient
    avg_dice = total_dice / num_samples if num_samples > 0 else 0.0
    print(f"Average Dice Coefficient on test set: {avg_dice:.4f}")

    # Save predicted masks to CSV
    pred_csv = output_dir / 'output.csv'
    pred_df = pd.DataFrame(pred_results, columns=['id', 'segmentation'])
    pred_df.to_csv(pred_csv, index=False)
    print(f"Predicted masks CSV saved: {pred_csv}")

    # Save original masks to CSV
    true_csv = output_dir / 'original_masks.csv'
    true_df = pd.DataFrame(true_results, columns=['id', 'segmentation'])
    true_df.to_csv(true_csv, index=False)
    print(f"Original masks CSV saved: {true_csv}")

    return avg_dice

In [ ]:
infer_and_evaluate(model_loaded, test_loader, device, "/kaggle/working/csvs")